<a href="https://colab.research.google.com/github/mcfvdesigns/Site-Turnover-Defects/blob/main/Baseline_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q ultralytics pyyaml

In [12]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [13]:
from ultralytics import YOLO

# 1. Load an un-finetuned, pre-trained base model
baseline_model = YOLO("yolo11n.pt")

# 2. Run baseline inference on test images
results = baseline_model.predict(
    source=str(dataset_root / "test/images"),
    save=True,
    conf=0.25
)

# 3. Evaluate baseline zero-shot performance on the validation split
metrics = baseline_model.val(data=str(cfg_path))
print("Baseline mAP50-95:", metrics.box.map)


image 1/5 /content/dataset/test/images/39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg: 640x640 (no detections), 8.1ms
image 2/5 /content/dataset/test/images/3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg: 640x640 (no detections), 9.3ms
image 3/5 /content/dataset/test/images/3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg: 640x640 (no detections), 13.5ms
image 4/5 /content/dataset/test/images/5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg: 640x640 1 motorcycle, 1 bed, 16.3ms
image 5/5 /content/dataset/test/images/6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67ae04.jpg: 640x640 1 person, 11.7ms
Speed: 1.3ms preprocess, 11.8ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict-3
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO1

In [14]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)


{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [15]:
from ultralytics import YOLO

# 1. Initialize a fresh YOLO11 model
model = YOLO("yolo11n.pt")

# 2. Train the model on your dataset using the config path defined in Cell 2
train_results = model.train(
    data=str(cfg_path),
    epochs=50,
    imgsz=640,
    plots=True
)

# 3. Evaluate the newly fine-tuned model on the validation set
val_metrics = model.val(data=str(cfg_path))
print("Fine-tuned mAP50-95:", val_metrics.box.map)

Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=None, opset=None, optimize=Fa

In [17]:
# Run inference on a new batch of test images
inference_results = model.predict(
    source=str(dataset_root / "test/images"),
    save=True,
    conf=0.25
)

print("Inference completed and results saved.")


image 1/5 /content/dataset/test/images/39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg: 640x640 (no detections), 9.1ms
image 2/5 /content/dataset/test/images/3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg: 640x640 (no detections), 8.9ms
image 3/5 /content/dataset/test/images/3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg: 640x640 (no detections), 9.0ms
image 4/5 /content/dataset/test/images/5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg: 640x640 (no detections), 9.1ms
image 5/5 /content/dataset/test/images/6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67ae04.jpg: 640x640 (no detections), 8.1ms
Speed: 1.3ms preprocess, 8.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict-4
Inference completed and results saved.
